# Creditworthiness ML Walkthrough
### Reusing the existing BI-ML implementation without changing it

## 1. Project introduction and research objectives

This notebook is a **research companion** to the `loan_project` repository. It walks through the
classification study that already exists in the project, using the project's own modules,
fitted pipelines and exported artifacts.

**Source of truth:** `feature_engineering_pipeline.py`, `extra_features_experiment.py`,
`threshold_tuning.py`, `shap_explain.py` and `portal_core.py`. This notebook does **not**
rewrite, replace or retune them. It imports them, reads their outputs, and explains the results.

### Research objectives

1. Reconstruct how the three raw Zindi files become the 37-feature model matrix.
2. Document the preprocessing rules (missingness, categorical encoding, train-only imputation).
3. Inspect the two deployed classifiers — Logistic Regression and Random Forest — as fitted.
4. Examine the class-imbalance problem and the role (and current absence) of SMOTE.
5. Reproduce the threshold operating points and confusion matrices without re-tuning.
6. Interpret the models with SHAP using the project's own explanation logic.
7. Compare the models and record findings, limitations and reproducibility notes.

### What this notebook deliberately does not do

- It does not retrain either model (no `fit`, no `cross_val_predict`, no `threshold_tuning.main()`).
- It does not overwrite `thresholds.json`, `feature_columns.json`, `encoding_maps.json`, the
  `.joblib` pipelines or any CSV.
- It does not modify the Flask/React application, the database, or the original scripts.
- It does not download the Zindi dataset. The data are **not redistributed** in the repository;
  you must obtain them yourself under the competition's terms.

## 2. Environment setup and dependency verification

### Quick start in Google Colab

1. **Open this notebook in Colab** (File → Upload notebook, or open it from GitHub).
2. Run the setup cells below. If the repository is not present, the first cell clones it.
3. Upload the generated artifacts that are **git-ignored** and therefore absent from a fresh clone.
   Colab's Files pane uploads to `/content` by default; the setup cell detects the data directory
   automatically, so you do not need to move the files into the repository folder.

| File | Why it is needed |
|---|---|
| `model_ready_extended.csv` | 37 features + `good_bad_flag`; input to modelling and SHAP |
| `test_set.csv` | the held-out 30% split used by the original evaluation |
| `thresholds.json`, `feature_columns.json` | deployed decision thresholds and feature order |
| `logistic_regression_pipeline.joblib`, `random_forest_pipeline.joblib` | fitted pipelines |
| `threshold_comparison.csv`, `feature_importance.csv` | canonical metrics and importances |
| `shap_global.csv`, `shap_local.csv` | precomputed SHAP tables (optional but recommended) |
| `logistic_regression_test_predictions.csv`, `random_forest_test_predictions.csv` | version-independent fallback for evaluation |

4. Optionally upload the three raw files (`traindemographics.csv`, `trainperf.csv`,
   `trainprevloans.csv`) to exercise the feature-engineering sections.

> **No credentials are required.** Do not upload `.env`, `creditscore.db` or
> `applications_log.csv`; they contain secrets, password hashes or applicant records.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Roni-creator/loan-project.git"

def find_project_root():
    here = Path.cwd().resolve()
    candidates = [
        here,
        *here.parents,
        Path("/content/loan-project"),
        Path("/content/loan_project"),
    ]
    for candidate in candidates:
        if (candidate / "threshold_tuning.py").exists() and (candidate / "portal_core.py").exists():
            return candidate
    return None

ROOT = find_project_root()
if ROOT is None:
    destination = Path("/content/loan-project")
    print(f"Repository not found locally. Cloning into {destination} ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(destination)], check=False)
    ROOT = find_project_root()

if ROOT is None:
    raise RuntimeError(
        "Could not locate the loan_project repository. In Colab, clone it with "
        f"`!git clone {REPO_URL}` or upload the project files, then rerun this cell."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def find_data_dir():
    # Prefer the repo root, then Colab's default upload folder, then Drive.
    candidates = [
        ROOT,
        Path("/content"),
        Path.cwd(),
        Path("/content/drive/MyDrive"),
    ]
    for candidate in candidates:
        if (candidate / "model_ready_extended.csv").exists():
            return candidate
    for candidate in candidates:
        if (candidate / "trainperf.csv").exists():
            return candidate
    return ROOT


DATA_DIR = find_data_dir()
os.chdir(DATA_DIR)

print("Project root :", ROOT)
print("Data dir     :", DATA_DIR)
print("Working dir  :", Path.cwd())

In [ ]:
import importlib.util
import subprocess
import sys

# Only the research stack. Flask/Streamlit/bcrypt from requirements.txt are for the web app.
RESEARCH_DEPS = {
    "sklearn": "scikit-learn",
    "imblearn": "imbalanced-learn",
    "shap": "shap",
    "matplotlib": "matplotlib",
    "statsmodels": "statsmodels",
    "joblib": "joblib",
    "pandas": "pandas",
    "numpy": "numpy",
}

missing = [pkg for mod, pkg in RESEARCH_DEPS.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing missing research dependencies:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All research dependencies are already importable.")

print("\nNote: if pip replaces NumPy/SciPy, restart the Colab runtime before continuing.")

In [ ]:
import platform
from importlib.metadata import PackageNotFoundError, version

import pandas as pd

# Versions recorded in the environment that produced the committed artifacts.
RECORDED_LOCAL_ENV = {
    "python": "3.12.6",
    "scikit-learn": "1.9.1",
    "pandas": "3.0.6",
    "numpy": "2.5.3",
    "imbalanced-learn": "0.14.2",
    "joblib": "1.6.0",
    "shap": "0.54.1",
    "statsmodels": "0.15.0",
}


def installed(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not installed"


rows = []
for pkg, recorded in RECORDED_LOCAL_ENV.items():
    current = platform.python_version() if pkg == "python" else installed(pkg)
    rows.append({
        "package": pkg,
        "artifact_env": recorded,
        "this_session": current,
        "status": "match" if current == recorded else "different",
    })

versions = pd.DataFrame(rows)
display(versions)

sklearn_row = versions.loc[versions["package"] == "scikit-learn", "status"].iloc[0]
if sklearn_row != "match":
    print("\nWARNING: scikit-learn differs from the artifact-producing environment.")
    print("joblib does not guarantee cross-version compatibility. If a pipeline fails to load,")
    print("this notebook falls back to the saved *_test_predictions.csv files and still runs.")

## 3. Dataset loading and understanding

The study uses the **Zindi Data Science Nigeria Challenge #1** loan-default dataset. It is not
redistributed in the repository: `README.md` points to the competition page, and `.gitignore`
excludes `*.csv`, `*.joblib` and databases.

| Raw file | Grain | Role |
|---|---|---|
| `traindemographics.csv` | one row per customer (with a few duplicates) | demographics: bank, account type, employment, birth date, GPS |
| `trainperf.csv` | one row per current loan | loan amount, term, total due, target (`good_bad_flag`) |
| `trainprevloans.csv` | one row per previous loan | repayment dates used to engineer willingness-to-pay features |

The three files are joined on `customerid` by `feature_engineering_pipeline.py`. The target is
**Bad = 0** (default) and **Good = 1** (repaid), and the positive class is the majority.

The dataset discovery cell below reports exactly what is available in this runtime. Missing raw
files do not block model evaluation; missing generated artifacts do, because the notebook will
not retrain models on your behalf.

In [ ]:
import pandas as pd
from pathlib import Path

RAW_FILES = {
    "traindemographics.csv": "customer demographics (Zindi; not in git)",
    "trainperf.csv": "current loans + target (Zindi; not in git)",
    "trainprevloans.csv": "prior loans, used for behavioural features (Zindi; not in git)",
}

ARTIFACT_FILES = {
    "model_ready_dataset.csv": "base engineered dataset",
    "model_ready_extended.csv": "37 features + target; model input",
    "test_set.csv": "held-out 30% test split",
    "logistic_regression_pipeline.joblib": "fitted LR pipeline",
    "random_forest_pipeline.joblib": "fitted RF pipeline",
    "thresholds.json": "deployed P(Good) cut-offs",
    "feature_columns.json": "canonical feature order",
    "encoding_maps.json": "raw-category -> training-code maps (tracked in git)",
    "threshold_comparison.csv": "canonical default vs tuned metrics",
    "feature_importance.csv": "LR coefficients and RF importances",
    "shap_global.csv": "precomputed global SHAP table",
    "shap_local.csv": "precomputed per-borrower SHAP table",
    "logistic_regression_test_predictions.csv": "saved LR test scores",
    "random_forest_test_predictions.csv": "saved RF test scores",
}


def inventory(mapping):
    rows = []
    for name, role in mapping.items():
        path = Path(name)
        rows.append({
            "file": name,
            "role": role,
            "present": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else None,
        })
    return pd.DataFrame(rows)


print("RAW INPUTS (must be obtained from Zindi)")
display(inventory(RAW_FILES))
print("GENERATED ARTIFACTS (git-ignored; produced by build_all.py / threshold_tuning.py)")
display(inventory(ARTIFACT_FILES))

In [ ]:
import feature_engineering_pipeline as fe

raw_available = all(Path(name).exists() for name in RAW_FILES)
model_ready_path = Path("model_ready_extended.csv")

if raw_available:
    demo, perf, prev = fe.load_raw(str(DATA_DIR))
    print("Raw shapes:")
    print("  traindemographics:", demo.shape)
    print("  trainperf       :", perf.shape)
    print("  trainprevloans  :", prev.shape)

    behavioural = fe.engineer_behavioural_features(prev)
    merged_raw = fe.merge_datasets(demo, perf, behavioural)
    print("\nMerged raw shape:", merged_raw.shape)
else:
    print("Raw Zindi files are not present in this runtime.")
    print("Upload them to run the full feature-engineering walkthrough.")
    print("Model evaluation, threshold analysis and SHAP only need the generated artifacts.")

if not model_ready_path.exists():
    raise FileNotFoundError(
        "model_ready_extended.csv is missing from the detected data directory "
        f"({Path.cwd()}). Upload it there (Colab: /content) or generate it with "
        "`python build_all.py` after obtaining the raw data."
    )

model_ready = pd.read_csv(model_ready_path)
print("\nmodel_ready_extended.csv:", model_ready.shape)
display(model_ready.head(3))

In [ ]:
target = "good_bad_flag"


def bad_share(series):
    if pd.api.types.is_numeric_dtype(series):
        return float((pd.to_numeric(series, errors="coerce") == 0).mean())
    return float(series.astype(str).str.strip().str.lower().isin(["bad", "0"]).mean())


balance = (
    model_ready[target]
    .astype(str)
    .value_counts()
    .to_frame("count")
    .assign(share=lambda d: (d["count"] / len(model_ready)).round(4))
)
print("Class balance in model_ready_extended.csv:")
display(balance)
print(f"Bad share: {bad_share(model_ready[target]):.2%}")

missing = model_ready.isna().sum().sort_values(ascending=False)
print("\nColumns with missing values (the in-pipeline imputer handles these):")
display(missing[missing > 0].to_frame("missing").assign(
    share=lambda d: (d["missing"] / len(model_ready)).round(4)
))

print("\nDtype counts:", model_ready.dtypes.value_counts().to_dict())

if Path("test_set.csv").exists():
    test_set = pd.read_csv("test_set.csv")
    print("\ntest_set.csv:", test_set.shape, f"| Bad share: {bad_share(test_set[target]):.2%}")

## 4. Data preprocessing and feature engineering

The original pipeline is a chain of small, reviewable functions in
`feature_engineering_pipeline.py`:

1. **Behavioural aggregation** (`engineer_behavioural_features`): one row per customer with
   `prior_loan_count`, `prior_default_rate` (>3 days late), `rollover_count` (>30 days late) and
   `days_to_first_repayment`.
2. **Merge** (`merge_datasets`): anchored on `trainperf`, left-joined to deduplicated demographics
   and the behavioural block. Customers with no history keep null behavioural values and get
   `prior_loan_count = 0`.
3. **High-missingness drop** (`drop_high_missingness`): `bank_branch_clients`,
   `level_of_education_clients` and `referredby` are dropped (>50% missing).
4. **Unknown category** (`fill_unknown_categories`): every text column gets `"Unknown"`; this is a
   fixed rule, safe before the split.
5. **Encoding** (`encode_categoricals`): label encoding for binary columns, one-hot with
   `drop_first=True` for `bank_name_clients`; the mapping is exported to `encoding_maps.json`.
6. **Pruning** (`drop_non_feature_columns`, `drop_collinear_columns`): IDs, raw dates and GPS are
   removed; `totaldue` is dropped (VIF > 240 against `loanamount`).
7. **Extended features** (`extra_features_experiment.build_extra_features`): `interest_ratio`,
   application hour/day, age, prior-loan amount/term/delay/on-time aggregates and
   `loan_to_prior_avg`.

Numeric NaNs are **not** imputed here. They are filled with **training-fold medians** inside the
sklearn pipeline, so no test statistic leaks into fitting.

The next cell calls the existing functions in memory. `encoding_maps.json` is passed
`maps_path=None` so the notebook cannot overwrite the tracked configuration file.

In [ ]:
if raw_available:
    print("Behavioural features (one row per customer):", behavioural.shape)
    display(behavioural.head())
    display(
        behavioural[
            ["prior_loan_count", "prior_default_rate", "rollover_count", "days_to_first_repayment"]
        ].describe().round(4)
    )

    merged = fe.merge_datasets(demo, perf, behavioural)
    columns_before = merged.shape[1]
    merged = fe.drop_high_missingness(merged)
    print(f"\nHigh-missingness drop: {columns_before} -> {merged.shape[1]} columns")
    print("Dropped:", [c for c in fe.HIGH_MISSINGNESS_COLUMNS if c not in merged.columns])

    merged = fe.fill_unknown_categories(merged)
    encoded = fe.encode_categoricals(merged, maps_path=None)  # do NOT overwrite encoding_maps.json
    bank_dummies = [c for c in encoded.columns if c.startswith("bank_name_clients_")]

    print("Encoded shape:", encoded.shape)
    print("Label-encoded columns:", fe.BINARY_CATEGORICAL_COLUMNS)
    print("Bank one-hot columns:", len(bank_dummies))
    print("encoding_maps.json was left untouched.")
else:
    print("Skipped: raw Zindi files are absent.")

In [ ]:
import extra_features_experiment as efe

if raw_available:
    extra = efe.build_extra_features(str(DATA_DIR))
    print("Extra feature block:", extra.shape)
    display(extra.head(3))
    print("\nAdded features:", list(extra.columns))
    print("Infinity values are replaced with NaN; medians are learned inside the model pipeline.")

    try:
        vif_frame = fe.fill_unknown_categories(
            fe.drop_high_missingness(fe.merge_datasets(demo, perf, behavioural))
        )
        print("\nVIF check from the existing helper (loanamount, totaldue, termdays):")
        display(fe.check_vif(vif_frame))
    except Exception as exc:
        print("\nVIF check skipped:", type(exc).__name__, exc)
else:
    print("Skipped: raw Zindi files are absent.")
    if Path("model_ready_dataset.csv").exists():
        base_columns = set(pd.read_csv("model_ready_dataset.csv").columns)
        extra_columns = [c for c in model_ready.columns if c not in base_columns]
        print("Extended-only columns visible in model_ready_extended.csv:", extra_columns)

## 5. Existing classification methodology

The canonical script `threshold_tuning.py` fixes every modelling choice:

- **Target convention:** `Bad = 0`, `Good = 1`; `P(Good) = predict_proba(X)[:, 1]`.
- **Decision rule:** predict Good when `P(Good) >= threshold`; otherwise Bad.
- **Split:** one stratified 70/30 holdout (`random_state=42`) — 3,057 train / 1,311 test rows.
- **Preprocessing inside the pipeline:** median imputer → standard scaler → classifier.
- **Logistic Regression:** L2 penalty (default), `class_weight="balanced"`, `max_iter=1000`.
- **Random Forest:** 300 trees, `random_state=42`, no class weighting.
- **Threshold tuning:** 5-fold `StratifiedKFold` out-of-fold predictions on the **training set
  only**; the threshold maximises F1 of the **Bad** class over a 0.05–0.95 grid (step 0.005).
- **Test set:** scored once for the final report.
- **SMOTE:** `USE_SMOTE = False` in the canonical pipeline.

The next cell imports the module and reads these values directly, rather than restating them in
notebook code. Importing the module executes only its configuration — `main()` is guarded by
`if __name__ == "__main__"`, so nothing is trained.

In [ ]:
import threshold_tuning as tt

print("Canonical configuration (read from threshold_tuning.py):")
for key in ["DATA_PATH", "TARGET_COL", "RANDOM_STATE", "TEST_SIZE", "USE_SMOTE", "MODEL_FILES"]:
    print(f"  {key}: {getattr(tt, key)}")
print(f"  THRESHOLD_GRID: {len(tt.THRESHOLD_GRID)} values from "
      f"{tt.THRESHOLD_GRID[0]} to {tt.THRESHOLD_GRID[-1]}")

print("\nUnfitted pipeline templates:")
for name in tt.MODEL_FILES:
    template = tt.build_pipeline(name)
    print(f"  {name}: {[step[0] for step in template.steps]}")
    print(f"    classifier: {template.steps[-1][1]}")

X, y = tt.load_data(str(Path("model_ready_extended.csv")))
X_train, X_test, y_train, y_test = tt.train_test_split(
    X, y, test_size=tt.TEST_SIZE, stratify=y, random_state=tt.RANDOM_STATE
)

print("\nFeature matrix:", X.shape)
print("Split:", X_train.shape, "train /", X_test.shape, "test")
print(f"Bad share — train: {(y_train == tt.BAD).mean():.2%} | test: {(y_test == tt.BAD).mean():.2%}")

In [ ]:
import json
import warnings

import joblib

try:
    from sklearn.exceptions import InconsistentVersionWarning
except Exception:  # very old sklearn
    InconsistentVersionWarning = Warning

thresholds = {}
if Path("thresholds.json").exists():
    thresholds = json.loads(Path("thresholds.json").read_text(encoding="utf-8"))
else:
    print("WARNING: thresholds.json is missing; falling back to 0.50 for both models.")

feature_columns = []
if Path("feature_columns.json").exists():
    feature_columns = json.loads(Path("feature_columns.json").read_text(encoding="utf-8"))

THR = {name: float(thresholds.get(name, 0.50)) for name in tt.MODEL_FILES}
print("Deployed thresholds:", THR)
print("feature_columns.json matches the feature matrix:", feature_columns == list(X.columns))

PIPELINES = {}
for name, filename in tt.MODEL_FILES.items():
    path = Path(filename)
    if not path.exists():
        print(f"{name}: MISSING {filename} — upload it or regenerate it with build_all.py.")
        continue
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            pipe = joblib.load(path)
        version_warnings = [
            str(w.message) for w in caught if issubclass(w.category, InconsistentVersionWarning)
        ]
        PIPELINES[name] = pipe
        suffix = f" | VERSION WARNING: {version_warnings[0]}" if version_warnings else ""
        print(f"{name}: loaded OK{suffix}")
        print(f"  classes={list(pipe.classes_)} | n_features_in_={pipe.n_features_in_}")
    except Exception as exc:
        print(f"{name}: could not load ({type(exc).__name__}: {exc}).")
        print("  Falling back to the saved test predictions for evaluation.")

## 6. Logistic Regression

The deployed Logistic Regression is a linear model on scaled features with balanced class
weights. Its coefficients are log-odds of **Good**: positive values push toward repayment,
negative values push toward default. Because the inputs are standardised, coefficient magnitudes
are comparable across numeric features (not across one-hot groups).

The cell below reads the **already fitted** estimator from the `.joblib` pipeline. It performs
inference on the held-out test rows only — no fitting, no threshold search. If the artifact cannot
be loaded in this Colab runtime, it falls back to the saved prediction CSV and the exported
coefficient table.

In [ ]:
import numpy as np


def saved_predictions(model_name):
    filename = model_name.lower().replace(" ", "_") + "_test_predictions.csv"
    path = Path(filename)
    return pd.read_csv(path) if path.exists() else None


def probabilities(model_name):
    # Live inference when the pipeline loads, otherwise the saved out-of-sample scores.
    if model_name in PIPELINES:
        return PIPELINES[model_name].predict_proba(X_test)[:, 1], "live pipeline"
    saved = saved_predictions(model_name)
    if saved is not None and len(saved) == len(X_test):
        return saved["p_good"].to_numpy(), "saved predictions CSV"
    raise RuntimeError(f"No usable pipeline or saved predictions for {model_name}.")


model_name = "Logistic Regression"
pipe = PIPELINES.get(model_name)

if pipe is not None:
    clf = pipe.named_steps["clf"]
    print("Fitted estimator:", clf)
    print("Intercept:", clf.intercept_.round(4))
    coefficients = pd.Series(clf.coef_[0], index=X.columns)
    coefficients = coefficients.reindex(coefficients.abs().sort_values(ascending=False).index)
    print("\nTop 10 coefficients on scaled features (+ pushes toward Good):")
    display(coefficients.head(10).to_frame("coefficient").round(4))
else:
    print("Pipeline not loaded; showing the exported coefficients instead.")
    importance = pd.read_csv("feature_importance.csv").dropna(subset=["lr_coefficient"])
    display(importance.nsmallest(5, "lr_coefficient")[
        ["feature", "lr_coefficient", "lr_direction", "driver_type"]
    ])
    display(importance.nlargest(5, "lr_coefficient")[
        ["feature", "lr_coefficient", "lr_direction", "driver_type"]
    ])

p_good_lr, lr_source = probabilities(model_name)
print(f"\nScored {len(p_good_lr)} test borrowers using the {lr_source}.")
print("P(Good) summary:", pd.Series(p_good_lr).describe().round(4).to_dict())

## 7. Random Forest

The Random Forest uses 300 unweighted trees. Unlike Logistic Regression, it can represent
non-linearities and interactions (for example, the joint effect of loan amount and prior delay),
but it is less transparent and can be conservative at high P(Good) thresholds.

Feature importance below is impurity-based (`feature_importances_`) and is **not** a causal
effect. The SHAP section provides a second, additive view.

In [ ]:
model_name = "Random Forest"
pipe = PIPELINES.get(model_name)

if pipe is not None:
    clf = pipe.named_steps["clf"]
    print("Fitted estimator:", clf)
    importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("\nTop 10 impurity-based importances:")
    display(importances.head(10).to_frame("importance").round(4))
    print("\nBottom 5:")
    display(importances.tail(5).to_frame("importance").round(4))
else:
    print("Pipeline not loaded; showing the exported importances instead.")
    importance = pd.read_csv("feature_importance.csv").sort_values("rf_rank")
    display(importance.head(10)[["feature", "rf_importance", "driver_type"]])

p_good_rf, rf_source = probabilities(model_name)
print(f"\nScored {len(p_good_rf)} test borrowers using the {rf_source}.")
print("P(Good) summary:", pd.Series(p_good_rf).describe().round(4).to_dict())

## 8. SMOTE and class imbalance

The target is imbalanced: roughly **78% Good / 22% Bad**. A model that always predicts Good
would be 78% accurate while catching no defaults, so accuracy is not the right headline metric.
The project therefore tunes and reports **Bad-class F1**, precision, recall, ROC-AUC and PR-AUC.

Two different treatments exist in the repository:

- **Canonical training** (`threshold_tuning.py`): `USE_SMOTE = False`. Logistic Regression uses
  `class_weight="balanced"`; Random Forest uses no weighting. Threshold tuning is the mechanism
  used to reach the desired Bad-class operating point.
- **Feature experiment** (`extra_features_experiment.py`): applies SMOTE to training folds when
  comparing baseline vs extended feature sets, with different model settings. Those experimental
  numbers are not the deployed model.

The illustration below resamples the training matrix **in memory only**. It fits no classifier
and writes no file. It exists to show what SMOTE would do to the class counts.

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer

counts = pd.Series(y).value_counts().rename(index={0: "Bad", 1: "Good"}).to_frame("count")
counts["share"] = (counts["count"] / len(y)).round(4)
print("Full modelling population:")
display(counts)

print("Canonical USE_SMOTE:", tt.USE_SMOTE)
print("Canonical steps:", [step[0] for step in tt.build_pipeline("Logistic Regression").steps])
print("SMOTE, if enabled, sits inside the pipeline after scaling and before the classifier,")
print("so it is applied to training folds only — never to validation or test rows.")

RUN_SMOTE_ILLUSTRATION = True
if RUN_SMOTE_ILLUSTRATION:
    imputer = SimpleImputer(strategy="median")
    X_train_imputed = pd.DataFrame(
        imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index
    )
    before = pd.Series(y_train).value_counts().sort_index()
    X_resampled, y_resampled = SMOTE(random_state=tt.RANDOM_STATE).fit_resample(
        X_train_imputed, y_train
    )
    after = pd.Series(y_resampled).value_counts().sort_index()
    comparison = pd.DataFrame({"before": before, "after": after}).rename(
        index={0: "Bad", 1: "Good"}
    )
    print("\nIllustration only — no model is fitted and no artifact is written:")
    display(comparison)
    print("Resampled matrix shape:", X_resampled.shape)
else:
    print("SMOTE illustration disabled.")

## 9. Threshold optimization

The deployed decision is not a fixed 0.50 cut-off. `threshold_tuning.best_threshold` selects the
cut-off that maximises **Bad-class F1** on out-of-fold **training** predictions. The chosen values
are stored in `thresholds.json`:

| Model | P(Good) threshold |
|---|---|
| Logistic Regression | 0.555 |
| Random Forest | 0.750 |

The test set is used **once** for the final report. The curve below is a diagnostic replay on the
saved test scores to show the precision/recall trade-off; it does **not** re-tune anything and does
not modify `thresholds.json`. The canonical metrics remain those in `threshold_comparison.csv`.

A notable result: for Logistic Regression, the tuned threshold raises Bad recall (0.559 → 0.668)
but lowers Bad F1 (0.459 → 0.439) on this particular test split. That is expected when tuning on
out-of-fold training data — it is a documented generalisation gap, not a bug to fix here.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

print("Canonical thresholds:")
display(pd.Series(THR, name="threshold_P_good").to_frame())

if Path("threshold_comparison.csv").exists():
    comparison = pd.read_csv("threshold_comparison.csv")
    display(comparison)
else:
    comparison = None
    print("threshold_comparison.csv not found.")

y_bad_test = (y_test.to_numpy() == tt.BAD).astype(int)
fig, ax = plt.subplots(figsize=(8, 4))
for label, p_good in [("Logistic Regression", p_good_lr), ("Random Forest", p_good_rf)]:
    scores = [
        f1_score(y_bad_test, (p_good < threshold).astype(int), zero_division=0)
        for threshold in tt.THRESHOLD_GRID
    ]
    ax.plot(tt.THRESHOLD_GRID, scores, label=label)
    ax.axvline(THR[label], linestyle="--", linewidth=1, alpha=0.6)

ax.set_xlabel("P(Good) threshold")
ax.set_ylabel("Bad-class F1 (test, diagnostic)")
ax.set_title("Threshold trade-off — visualization only")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Model evaluation and confusion matrices

Metrics follow the project's own `bad_class_metrics` helper, with the Bad class treated as the
positive class. The confusion matrices use the deployed threshold for each model:

- **TP (Bad)** — defaulters correctly flagged.
- **FN (Bad)** — defaulters missed (the costly error in credit screening).
- **FP (Bad)** — good borrowers incorrectly flagged.
- **TN (Bad)** — good borrowers correctly cleared.

The naive baseline from `threshold_tuning.naive_baseline_row` — flag everyone as Bad — has
recall 1.0 but precision equal to the Bad prevalence (≈0.218) and F1 ≈0.358. Both models must be
read against that floor.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

metric_rows = []
for label, p_good in [("Logistic Regression", p_good_lr), ("Random Forest", p_good_rf)]:
    metric_rows.append({"model": label, **tt.bad_class_metrics(y_test.to_numpy(), p_good, THR[label])})

metrics = pd.DataFrame(metric_rows)
print("Held-out test metrics at the deployed thresholds:")
display(metrics.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (label, p_good) in zip(
    axes, [("Logistic Regression", p_good_lr), ("Random Forest", p_good_rf)]
):
    predicted_good = (p_good >= THR[label]).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predicted_good,
        labels=[1, 0],
        display_labels=["Good (1)", "Bad (0)"],
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(f"{label} — threshold {THR[label]:.3f}")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import auc, precision_recall_curve, roc_curve

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, p_good in [("Logistic Regression", p_good_lr), ("Random Forest", p_good_rf)]:
    fpr, tpr, _ = roc_curve(y_bad_test, 1 - p_good)
    precision, recall, _ = precision_recall_curve(y_bad_test, 1 - p_good)
    axes[0].plot(fpr, tpr, label=f"{label} (AUC={auc(fpr, tpr):.3f})")
    axes[1].plot(recall, precision, label=f"{label} (AP={auc(recall, precision):.3f})")

axes[0].plot([0, 1], [0, 1], "k:", linewidth=1)
axes[0].set(xlabel="False positive rate", ylabel="True positive rate (Bad)",
            title="ROC — Bad class")
axes[1].set(xlabel="Recall (Bad)", ylabel="Precision (Bad)",
            title="Precision–recall — Bad class")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. SHAP explainability

`shap_explain.py` is the project's explanation layer. Two conventions matter:

1. **Direction:** every SHAP value is expressed **toward Bad** — positive raises default risk,
   negative lowers it.
2. **Units differ by model:** Logistic Regression values are in log-odds of Bad; Random Forest
   values are in probability of Bad. Raw magnitudes are therefore **not comparable across
   models**. Compare `importance_share`, `rank` and `direction` instead.
3. **Bank grouping:** the 17 one-hot bank columns are summed into one `bank_name_clients` feature
   (exact, because SHAP values are additive).
4. **Additivity check:** the module verifies that `base_value + sum(SHAP) = model output` within
   `1e-3`; a failure indicates a sign, scaling or class-order problem.

The first cell reads the precomputed `shap_global.csv`. The second recomputes SHAP on a sample of
test rows with the existing `explain_frame` helper — again, inference only.

In [ ]:
shap_global_path = Path("shap_global.csv")
if shap_global_path.exists():
    shap_global = pd.read_csv(shap_global_path)
    print("shap_global.csv:", shap_global.shape)
    top_by_model = (
        shap_global.sort_values(["model", "rank"])
        .groupby("model")
        .head(8)
    )
    display(top_by_model[
        ["model", "rank", "feature", "driver_type", "importance_share", "direction"]
    ].round(4))

    pivot = shap_global.pivot_table(
        index="feature", columns="model", values="importance_share", aggfunc="first"
    ).fillna(0)
    top_features = pivot.assign(total=pivot.sum(axis=1)).nlargest(12, "total").drop(columns="total")

    ax = top_features.plot(kind="barh", figsize=(9, 6))
    ax.invert_yaxis()
    ax.set_xlabel("Mean |SHAP| importance share")
    ax.set_title("Top SHAP drivers by model")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("Cross-model rank agreement:")
    display(
        shap_global.drop_duplicates("feature")["models_agree"]
        .value_counts()
        .to_frame("features")
    )
else:
    print("shap_global.csv not found; run the live SHAP cell below.")

In [ ]:
LIVE_SHAP = True
SHAP_SAMPLE = 200

if LIVE_SHAP and PIPELINES:
    try:
        import shap_explain as se

        background = X_train.sample(min(100, len(X_train)), random_state=tt.RANDOM_STATE)
        X_shap = X_test.sample(min(SHAP_SAMPLE, len(X_test)), random_state=tt.RANDOM_STATE)
        live_results = {}

        for label, pipe in PIPELINES.items():
            res = se.explain_frame(pipe, X_shap, background)
            live_results[label] = res
            table = se.build_global(label, res)
            print(f"{label}: scale = {res['scale']} | additivity gap = {res['gap']:.2e}")
            display(table.head(8)[
                ["rank", "feature", "driver_type", "importance_share", "direction"]
            ].round(4))

        if "Logistic Regression" in live_results:
            res = live_results["Logistic Regression"]
            risk_index = int(np.argmax(res["p_bad"]))
            local = pd.DataFrame({
                "feature": res["names"],
                "value": res["value_text"][risk_index],
                "shap_toward_bad": res["grouped_values"][risk_index],
            })
            local["abs"] = local["shap_toward_bad"].abs()
            print(
                f"\nLocal explanation — highest-risk sampled borrower "
                f"(row {X_shap.index[risk_index]}, P(Bad) = {res['p_bad'][risk_index]:.3f}):"
            )
            display(local.nlargest(8, "abs").drop(columns="abs").round(4))
    except Exception as exc:
        print(f"Live SHAP skipped: {type(exc).__name__}: {exc}")
        print("The saved shap_global.csv / shap_local.csv remain the reference explanation.")
else:
    print("Live SHAP disabled or no pipelines were loaded.")

## 12. Comparative model analysis

The two models answer the same question but optimize different things:

- **Logistic Regression** is linear in the scaled features, balanced via `class_weight`, and easy
  to audit. Its coefficients map directly to the application's explanation panel.
- **Random Forest** captures non-linearities and interactions, but its probability scale is
  discrete (multiples of 1/300 before averaging) and its errors are dominated by the majority
  class because it is unweighted.

The comparison below uses the saved test scores, the canonical tuned operating points, the
exported feature rankings and (if available) the SHAP rank agreement. "Disagreement" is measured
on the final Good/Bad decision, not on the raw probabilities.

In [ ]:
predicted_lr = (p_good_lr >= THR["Logistic Regression"]).astype(int)
predicted_rf = (p_good_rf >= THR["Random Forest"]).astype(int)
actual = y_test.to_numpy()

disagreement = float((predicted_lr != predicted_rf).mean())
both_wrong = float(((predicted_lr != actual) & (predicted_rf != actual)).mean())
print(f"Decision disagreement: {disagreement:.1%} of test borrowers")
print(f"Both models wrong:     {both_wrong:.1%} of test borrowers")

display(pd.crosstab(
    pd.Series(predicted_lr, name="LR predicted Good"),
    pd.Series(predicted_rf, name="RF predicted Good"),
    margins=True,
))

if comparison is not None:
    tuned = comparison[comparison["setting"] == "tuned"].set_index("model")
    display(tuned[
        ["threshold_P_good", "precision_bad", "recall_bad", "f1_bad", "roc_auc", "pr_auc_bad"]
    ].round(3))
    ax = tuned[["precision_bad", "recall_bad", "f1_bad"]].plot(kind="bar", figsize=(8, 4))
    ax.set_title("Tuned operating point — Bad class")
    ax.set_ylabel("score")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

print("Correlation of P(Good) scores:", round(float(np.corrcoef(p_good_lr, p_good_rf)[0, 1]), 4))

if Path("feature_importance.csv").exists():
    feature_importance = pd.read_csv("feature_importance.csv")
    feature_importance["rank_gap"] = (
        feature_importance["lr_rank"] - feature_importance["rf_rank"]
    ).abs()
    print("\nLargest LR-vs-RF rank gaps:")
    display(feature_importance.nlargest(8, "rank_gap")[
        ["feature", "lr_rank", "rf_rank", "rank_gap", "driver_type"]
    ])

## 13. Research findings and limitations

### Findings

- **Class imbalance drives the metric choice.** With ~78% Good, accuracy is misleading. Bad-class
  F1, recall, PR-AUC and the confusion matrix are the meaningful outputs.
- **The models trade errors differently.** At their deployed thresholds, Logistic Regression
  achieves higher Bad recall; Random Forest is more conservative and has higher Bad precision.
  The right choice depends on the relative cost of a missed defaulter versus a rejected good
  applicant.
- **Willingness-to-pay features carry signal.** Prior repayment delay, prior default rate and
  rollover count appear among the top drivers in both coefficient and SHAP views. Ability/profile
  features (loan amount, application hour) also matter, but their ranking is model-dependent.
- **Threshold tuning is not a free win.** Optimising Bad F1 on out-of-fold training data improved
  recall but reduced test F1 for Logistic Regression on this split. The tuned threshold is a
  policy choice, not a guarantee of better test performance.
- **The models disagree on a non-trivial minority of applications.** That is why `portal_core.py`
  combines both votes and routes disagreement to manual review rather than forcing a decision.

### Limitations

- **Single holdout split.** The test set is scored once at 30%; there is no repeated nested CV,
  so metric uncertainty is not quantified in the exported tables.
- **No hyperparameter search in the canonical path.** Parameters are fixed in
  `threshold_tuning.build_pipeline`; the notebook respects them as-is.
- **Class imbalance handling is limited.** SMOTE is disabled in the deployed pipeline. The SMOTE
  experiment in `extra_features_experiment.py` uses different settings and is not the deployed
  model.
- **Data provenance.** The training data are Nigerian micro-loans from Zindi and are not
  redistributed. The portal labels amounts in KES, but that is a display relabelling — no currency
  conversion or recalibration occurred.
- **Excluded information.** IDs, raw dates, GPS and three high-missingness columns are dropped;
  the model therefore cannot use location or education signals.
- **SHAP units are model-specific.** Cross-model comparison must use ranks or importance shares,
  never raw SHAP magnitudes.
- **Artifact version sensitivity.** The `.joblib` files were produced with scikit-learn 1.9.1 and
  pandas 3.0.6. Loading them under different versions may warn or fail; the saved prediction CSVs
  provide a version-independent evaluation fallback.
- **No causal claims.** Coefficients, impurity importances and SHAP values describe model
  behaviour, not causal effects of changing an applicant's circumstances.
- **Sensitive data.** `creditscore.db` (password hashes), `applications_log.csv` (applicant data)
  and raw datasets are git-ignored and must never be committed or pasted into a public notebook.

### What was preserved

The original scripts, parameters, thresholds, datasets and application code were left unchanged.
This notebook only imports modules, loads artifacts and performs read-only inference.

In [ ]:
import hashlib
import platform
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import pandas as pd


def package_version(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not installed"


def short_hash(filename, limit_mb=50):
    path = Path(filename)
    if not path.exists():
        return None
    if path.stat().st_size > limit_mb * 1024 * 1024:
        return f"skipped (>{limit_mb} MB)"
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()[:16]


print("Runtime :", platform.platform())
print("Python  :", platform.python_version())
for pkg in ["scikit-learn", "pandas", "numpy", "imbalanced-learn", "joblib", "shap", "statsmodels"]:
    print(f"{pkg:16s}: {package_version(pkg)}")

manifest_files = [
    "model_ready_extended.csv",
    "test_set.csv",
    "thresholds.json",
    "feature_columns.json",
    "encoding_maps.json",
    "feature_importance.csv",
    "threshold_comparison.csv",
    "shap_global.csv",
    "logistic_regression_pipeline.joblib",
    "random_forest_pipeline.joblib",
]
display(pd.DataFrame([
    {"file": name, "exists": Path(name).exists(), "sha256_16": short_hash(name)}
    for name in manifest_files
]))

## Manual steps that still require Google Colab

1. **Obtain the data.** Download the three Zindi files under the competition's terms if you want
   to exercise Section 4. They are not distributed with this repository.
2. **Upload generated artifacts.** Because `.gitignore` excludes `*.csv` and `*.joblib`, a fresh
   clone has code only. Upload `model_ready_extended.csv`, `test_set.csv`, the two `.joblib`
   pipelines, the JSON configuration files and (optionally) the exported metric/SHAP tables.
   Colab's Files pane puts them in `/content`, which the setup cell checks automatically.
3. **Restart the runtime if prompted** after dependency installation, then rerun from the top.
4. **Check the version table.** If scikit-learn differs materially from 1.9.1 and a pipeline will
   not load, use the saved prediction CSVs for evaluation or regenerate artifacts locally with
   `python build_all.py`. Regeneration retrains models and is therefore an explicit, opt-in action —
   this notebook never performs it.
5. **Do not upload secrets.** Leave `.env`, `creditscore.db`, `applications_log.csv` and raw
   applicant data out of Colab and out of version control.